# Week 13 — Conduct a small model audit

**Research task:** Compare two prompt frames across hosted and local models while preserving every condition and avoiding unsupported cultural explanations.

**Python introduced:** nested loops, parameter grids, lists of records and a table created only after the plain records are understood.

This is the runnable coding component. The full literature-led chapter and slides remain to be developed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session13/session13_model_audits.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the Python SDKs used by this notebook, downloads the public course repository and selects OpenRouter because Colab cannot reach the Ollama server on your computer. On a local machine it installs nothing silently: it checks that the notebook is using the course environment and gives the exact repair command if it is not.

The setup also adds the repository root to Python's import path. This is necessary for Week 4's supplied image utility and prevents a second kind of `ModuleNotFoundError` after the repository has been cloned.


In [ ]:
SESSION = "session13"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib.util as setup_importlib
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath("/content/GenAI_Soc2026")
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The course repository root could not be found. Start Jupyter from the "
            "GenAI_Soc2026 folder with: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

## Define two routes, two prompt frames and one output schema

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
routes = ["openrouter"] if IN_COLAB else ["openrouter", "ollama"]
frames = {
    "individual":"Evaluate the action in terms of individual choice.",
    "collective":"Evaluate the action in terms of collective obligation.",
}
scenario = "A worker declines an unpaid request to stay late."
schema = {"type":"object","properties":{"approval":{"type":"integer","minimum":0,"maximum":100},"explanation":{"type":"string"}},"required":["approval","explanation"],"additionalProperties":False}
audit_records = []
print("Routes available in this runtime:", routes)
if IN_COLAB:
    print("The Ollama rows are completed later in local JupyterLab or on the in-class machine.")


## Run every route-by-frame condition

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
for route in routes:
    for frame_name, frame_instruction in frames.items():
        prompt = frame_instruction + " Score approval from 0 to 100. Scenario: " + scenario
        messages = [{"role":"user","content":prompt}]
        if route == "openrouter":
            with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
                response = client.chat.send(model=HOSTED_MODEL,messages=messages,temperature=0,response_format={"type":"json_schema","json_schema":{"name":"audit_response","strict":True,"schema":schema}})
            raw_output = response.choices[0].message.content
            requested_model = HOSTED_MODEL
        else:
            response = ollama.chat(model=LOCAL_MODEL,messages=messages,format=schema,options={"temperature":0})
            raw_output = response.message.content
            requested_model = LOCAL_MODEL
        parsed = json.loads(raw_output)
        record = {"route":route,"model":requested_model,"frame":frame_name,"prompt":prompt,"raw_output":raw_output,"approval":parsed["approval"]}
        audit_records.append(record)
        print("Condition record:", record)

## Inspect the plain records before making a table

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
for record in audit_records:
    print(record["route"], record["frame"], record["approval"])

## Convert the already understood records into a small table

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
import pandas as pd
audit_table = pd.DataFrame(audit_records)
print(audit_table[["route","model","frame","approval"]])

# ONE CHANGE: replace "worker" with "manager" and rerun the complete grid.

## Methodological check

The grid identifies patterned output differences under declared conditions. Route, model size, provider, prompt and training differences remain entangled; do not label the pattern a cultural or ideological cause. Colab produces the OpenRouter rows only; the local Ollama rows must be added outside Colab.

## Recording

Trace one cell through both loops, make the role-word change, present one bounded observed difference and name at least two unresolved explanations.
